## Fine-Tuning and Deploying Custom Models on Amazon Bedrock

### Introduction:

This notebook demonstrates the process of fine-tuning a Cohere `command-light-text-v14` model on Amazon Bedrock for generating concise clinical notes from patient-doctor conversations.

The dataset used is a **combined version of the ACI-Bench training and validation datasets**:
- **Source**: [ACI-Bench: Ambient Clinical Intelligence Benchmark](http://github.com/microsoft/clinical_visit_note_summarization_corpus)
- **Contents**: Patient-doctor dialogues paired with concise clinical notes.

### Why a Smaller Dataset?
- Reduce the cost of fine-tuning.
- Simplify the workflow for demonstration purposes.

In production, you would typically need thousands of examples.

### Technical Prerequisites
- Access to Amazon Bedrock with Cohere `command-light-text-v14` enabled
- AWS SDK (boto3) configured with appropriate IAM permissions
- Python 3.8+

## Step 1: Install Required Libraries

In [ ]:
!pip install datasets boto3 pandas

import pandas as pd
import boto3
import json
import os

## Step 2: Data Preparation

Load the dataset and inspect its structure before formatting for fine-tuning.

In [ ]:
train_dataset_path = "dataset/data.csv"
train_dataset = pd.read_csv(train_dataset_path)

print(f"Total records: {len(train_dataset)}")
print(f"Columns: {train_dataset.columns.tolist()}")

### Format as JSONL for Fine-Tuning

Each row is converted to `{prompt, completion}` format required by Amazon Bedrock.

In [ ]:
output_file_name = 'clinical_notes_fine_tune.jsonl'
output_file_path = os.path.join('dataset', output_file_name)

with open(output_file_path, 'w') as outfile:
    for _, row in train_dataset.iterrows():
        entry = {
            "prompt": f"Summarize the following conversation.\n\n{row['dialogue']}",
            "completion": row['note']
        }
        json.dump(entry, outfile)
        outfile.write('\n')

print(f"Dataset saved to {output_file_path}")
print(json.dumps({"prompt": train_dataset.iloc[0]['dialogue'][:100] + '...', "completion": train_dataset.iloc[0]['note']}, indent=2))

## Step 3: Upload the Dataset to S3

In [ ]:
bucket_name = 'bedrock-finetuning-bucket-YOUR-SUFFIX'  # ← Change this
s3_key = output_file_name
region = 'us-east-1'

s3_client = boto3.client('s3', region_name=region)

# Create bucket if it doesn't exist
existing = {b['Name'] for b in s3_client.list_buckets()['Buckets']}
if bucket_name not in existing:
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})
    print(f"Created bucket: {bucket_name}")
else:
    print(f"Bucket {bucket_name} already exists.")

s3_client.upload_file(output_file_path, bucket_name, s3_key)
print(f"Uploaded to s3://{bucket_name}/{s3_key}")

## Step 4: List Available Fine-Tuning Models

In [ ]:
bedrock_region = 'us-east-1'
bedrock = boto3.client('bedrock', region_name=bedrock_region)

response = bedrock.list_foundation_models(byCustomizationType='FINE_TUNING')
for model in response['modelSummaries']:
    print(f"  ID: {model['modelId']}  |  Name: {model['modelName']}")

## Step 5: Create IAM Role for Fine-Tuning

In [ ]:
iam = boto3.client('iam')
role_name = 'BedrockFineTuningRole'

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "bedrock.amazonaws.com"}, "Action": "sts:AssumeRole"}]
}

try:
    resp = iam.create_role(RoleName=role_name, AssumeRolePolicyDocument=json.dumps(trust_policy),
                           Description='Role for Bedrock fine-tuning')
    role_arn = resp['Role']['Arn']
    print(f"Created role: {role_arn}")
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = iam.get_role(RoleName=role_name)['Role']['Arn']
    print(f"Using existing role: {role_arn}")

permission_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket"],
        "Resource": [f"arn:aws:s3:::{bucket_name}", f"arn:aws:s3:::{bucket_name}/*"]
    }]
}
iam.put_role_policy(RoleName=role_name, PolicyName='BedrockS3Policy',
                    PolicyDocument=json.dumps(permission_policy))
print("Attached S3 permissions.")

## Step 6: Submit the Fine-Tuning Job

In [ ]:
base_model_id = 'cohere.command-light-text-v14:7:4k'
job_name = 'cohere-clinical-notes-finetune-v1'
model_name = 'cohere-clinical-notes-tuned-v1'

bedrock.create_model_customization_job(
    customizationType='FINE_TUNING',
    jobName=job_name,
    customModelName=model_name,
    roleArn=role_arn,
    baseModelIdentifier=base_model_id,
    hyperParameters={
        'epochCount': '3',
        'batchSize': '16',
        'learningRate': '0.00005',
    },
    trainingDataConfig={'s3Uri': f's3://{bucket_name}/{s3_key}'},
    outputDataConfig={'s3Uri': f's3://{bucket_name}/finetuned/'}
)
print(f"Fine-tuning job '{job_name}' submitted!")

## Step 7: Monitor Job Status

In [ ]:
import time

while True:
    status = bedrock.get_model_customization_job(jobIdentifier=job_name)['status']
    print(f"[{time.strftime('%H:%M:%S')}] Status: {status}")
    if status in ('Completed', 'Failed', 'Stopped'):
        break
    time.sleep(60)

## Step 8: Perform Model Inference

> **Note**: Before running inference, you must purchase **Provisioned Throughput** for your custom model in the Amazon Bedrock Console under **Custom Models → Models → Purchase Provisioned Throughput**. Copy the resulting model ARN and paste it below.

In [ ]:
bedrock_runtime = boto3.client('bedrock-runtime', region_name=bedrock_region)

custom_model_arn = 'YOUR_PROVISIONED_MODEL_ARN'  # ← Replace with your ARN

test_prompt = """
[doctor] Good morning. How have you been feeling since your last visit?
[patient] I've had persistent fatigue and dizziness, especially when standing up quickly.
[doctor] Any shortness of breath? Heart racing?
[patient] Heart racing sometimes, no shortness of breath. I also missed a few beta-blocker doses.
[doctor] That could be contributing. Let's check your BP and do an EKG. Also, your last labs showed elevated cholesterol and borderline A1c.
[patient] What does that mean for me?
[doctor] We'll consider a statin and refer you to a nutritionist. I'm also adjusting your diuretic for leg swelling.
"""

body = {
    'prompt': f'Summarize the following conversation.\n\n{test_prompt}',
    'temperature': 0.3,
    'p': 0.9,
    'max_tokens': 150
}

try:
    response = bedrock_runtime.invoke_model(modelId=custom_model_arn, body=json.dumps(body))
    result = json.loads(response['body'].read().decode('utf-8'))
    print("Clinical Note Summary:")
    print(result['generations'][0]['text'])
except Exception as e:
    print(f"Error: {e}")

## Cleanup

> ⚠️ **Important**: To avoid ongoing charges, remove your provisioned throughput from the Amazon Bedrock Console under **Inference → Provisioned Throughput**.

In [ ]:
# Optional: Delete uploaded training data from S3
# s3_client.delete_object(Bucket=bucket_name, Key=s3_key)
# print(f"Deleted s3://{bucket_name}/{s3_key}")
print("Cleanup complete. Remember to remove Provisioned Throughput from the AWS Console.")